In [15]:
import os
os.chdir('/home/ubuntu/BackTime')

import torch, numpy as np
from trainer import Trainer
from dataset import load_raw_data
from run import parser_args
from pprint import pprint
config = parser_args()

In [16]:
DEVICE = torch.device('cuda:0')
data_config = config.Dataset
if not data_config.use_timestamps:
    train_mean, train_std, train_data_seq, test_data_seq = load_raw_data(data_config)
    train_data_stamps = test_data_stamps = None
else:
    train_mean, train_std, train_data_seq, test_data_seq, train_data_stamps, test_data_stamps = load_raw_data(data_config)

train_mean, train_std, train_data_seq.shape, test_data_seq.shape

# set attacked variables
spatial_poison_num = max(int(round(train_data_seq.shape[1] * config.alpha_s)), 1)

atk_vars = np.arange(train_data_seq.shape[1])
atk_vars = np.random.choice(atk_vars, size=spatial_poison_num, replace=False)
atk_vars = torch.from_numpy(atk_vars).long().to(DEVICE)
print('shape of attacked_variables', atk_vars.shape)

# load target pattern
target_pattern = config.Target_Pattern
target_pattern = torch.tensor(target_pattern).float().to(DEVICE) * train_std
print(target_pattern)

shape of attacked_variables torch.Size([107])
tensor([18.4551, 25.2422, 40.2460, 55.6397, 40.2460, 25.2422, 18.4551],
       device='cuda:0')


In [21]:
atk_vars

tensor([242,  94, 292, 145, 278,  47,  97, 170, 110, 267, 195, 320, 223,  91,
        265,  14, 127,  60,   6, 167, 302,  58, 313, 254, 179, 152, 270,  81,
        219,   7,   5, 118, 213,  31, 248,  27, 288,  10,  18, 352, 298,  53,
        306, 103,  24, 214,  38, 243,  17, 284, 303,   1, 349, 109, 171, 126,
        147, 114, 354, 196, 294, 124,  40, 271, 185, 168, 348, 226, 330, 184,
         13, 304,  49, 146, 356, 190,  86,  63, 139,  36,  32,  26, 135, 122,
         39, 316, 183,  73, 332, 253, 212, 257, 137,  35, 290, 202, 293, 210,
         96, 138,  87, 106, 350,  34, 331, 125, 261], device='cuda:0')

In [19]:
exp_trainer = Trainer(config, 
                        atk_vars, 
                        target_pattern, 
                        train_mean, 
                        train_std, 
                        train_data_seq, 
                        test_data_seq,
                        train_data_stamps, 
                        test_data_stamps, 
                        DEVICE)

with torch.no_grad():
    state = torch.load('./checkpoints/attacker_PEMS04.pth')
    # load target pattern
    target_pattern = config.Target_Pattern
    target_pattern = torch.tensor(target_pattern).float().to(DEVICE) * train_std

    exp_trainer = Trainer(config, 
                        atk_vars, 
                        target_pattern, 
                        train_mean, 
                        train_std, 
                        train_data_seq, 
                        test_data_seq,
                        train_data_stamps, 
                        test_data_stamps, 
                        DEVICE)
    exp_trainer.load_attacker(state)

    exp_trainer.test()

shape of data: torch.Size([358, 1, 15724])
shape of data: torch.Size([358, 1, 5242])
shape of data: torch.Size([358, 1, 5242])
shape of data: torch.Size([358, 1, 15724])
shape of data: torch.Size([358, 1, 5242])
shape of data: torch.Size([358, 1, 5242])


AssertionError: the batch graph size should be a multiple of the number of variables.

In [23]:
exp_trainer.test()

AssertionError: the batch graph size should be a multiple of the number of variables.